**BANK CUSTOMER CHURN PREDICTION**
###### - ANANYA PATANKAR


---



In [1]:
#import required libraries
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer,StandardScaler, MinMaxScaler, RobustScaler
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
import shap

import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")

**REBUILDING RAW DATA :**

The original dataset is reloaded and cleaned identically to notebook 02. This is to split it similarly to recover the exact train/test customer classifications.

In [2]:
df = pd.read_csv('bank churn dataset.csv')
df.drop(columns=['RowNumber', 'CustomerId', 'Surname'], inplace=True)
df.rename(columns={'Geography': 'Country', 'HasCrCard': 'HasCreditCard','IsActiveMember': 'IsActive', 'EstimatedSalary': 'Salary'}, inplace=True)
X_raw = df.drop(columns='Exited')
y_raw = df['Exited']

In [3]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw)

**BUILDING THE DEPLOYMENT PIPELINE :**

The feature engineering, categorical encoding, scaling, SMOTE, and the tuned LightGBM model is compiled into a single imblearn pipeline that can take raw input directly to a prediction. This is compiled into a single imblearn pipeline that can take raw input directly to prediction. Feature engineering is stored in a separte file (churn_utils.py) so that the saved pipleine can also be loaded elsewhere.


In [4]:
%%writefile churn_utils.py
FINAL_COLUMNS = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCreditCard', 'IsActive', 'Salary',
                 'ProductsRisk', 'ZeroBalance', 'AgeInactive', 'TenureAgeRatio', 'BalanceSalaryRatio',
                 'Country_Germany', 'Country_Spain', 'Gender_Male']

def engineer_features(df):
    df = df.copy()
    df['ProductsRisk'] = (df['NumOfProducts'] != 2).astype(int)
    df['ZeroBalance'] = (df['Balance'] == 0).astype(int)
    df['AgeInactive'] = df['Age'] * (1 - df['IsActive'])
    df['TenureAgeRatio'] = df['Tenure'] / df['Age']
    df['BalanceSalaryRatio'] = df['Balance'] / df['Salary']
    df['Country_Germany'] = (df['Country'] == 'Germany').astype(int)
    df['Country_Spain'] = (df['Country'] == 'Spain').astype(int)
    df['Gender_Male'] = (df['Gender'] == 'Male').astype(int)

    df = df.drop(columns=['Country', 'Gender'])
    return df[FINAL_COLUMNS]

Writing churn_utils.py


In [5]:
from churn_utils import engineer_features

feature_engineer = FunctionTransformer(engineer_features)

#scaling from notebook 02
preprocessor = ColumnTransformer(transformers=[('std', StandardScaler(), ['CreditScore', 'Age', 'Tenure']),
                                               ('minmax', MinMaxScaler(), ['NumOfProducts']),
                                               ('robust', RobustScaler(), ['Balance', 'Salary', 'AgeInactive', 'BalanceSalaryRatio']),
                                               ('keep', 'passthrough', ['HasCreditCard', 'IsActive', 'ProductsRisk', 'ZeroBalance',
                                                                        'TenureAgeRatio', 'Country_Germany', 'Country_Spain', 'Gender_Male'])])
#tuned hyperparameters from notebook 04
tuned_params = {'n_estimators': 280, 'max_depth': 6, 'learning_rate': 0.015315301378268475,
                'num_leaves': 102, 'min_child_samples': 90, 'subsample': 0.6232006436893751,
                'colsample_bytree': 0.741693755720684, 'subsample_freq': 1, 'random_state': 42,
                'deterministic': True, 'force_row_wise': True, 'verbose': -1}

pipeline = ImbPipeline(steps=[('engineer', feature_engineer),
                              ('scale', preprocessor),
                              ('smote', SMOTE(random_state=42)),
                              ('classifier', LGBMClassifier(**tuned_params))])

pipeline.fit(X_train_raw, y_train)

Pipeline(steps=[('engineer',
                 FunctionTransformer(func=<function engineer_features at 0x7f3c3cc08fe0>)),
                ('scale',
                 ColumnTransformer(transformers=[('std', StandardScaler(),
                                                  ['CreditScore', 'Age',
                                                   'Tenure']),
                                                 ('minmax', MinMaxScaler(),
                                                  ['NumOfProducts']),
                                                 ('robust', RobustScaler(),
                                                  ['Balance', 'Salary',
                                                   'AgeInactive',
                                                   'BalanceSalaryRatio']),
                                                 ('keep', 'passthrough',
                                                  ['HasC...
                                                   'Country_Germany',
                                                   'Country_Spain',
                                                   'Gender_Male'])])),
                ('smote', SMOTE(random_state=42)),
                ('classifier',
                 LGBMClassifier(colsample_bytree=0.741693755720684,
                                deterministic=True, force_row_wise=True,
                                learning_rate=0.015315301378268475, max_depth=6,
                                min_child_samples=90, n_estimators=280,
                                num_leaves=102, random_state=42,
                                subsample=0.6232006436893751, subsample_freq=1,
                                verbose=-1))])

Verify pipeline performance:

In [6]:
y_pred = pipeline.predict(X_test_raw)
y_proba = pipeline.predict_proba(X_test_raw)[:, 1]

print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score : {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_proba):.4f}")

Accuracy : 0.8500
Precision: 0.6230
Recall   : 0.6658
F1 Score : 0.6437
ROC-AUC  : 0.8652


The pipeline's performance matches the model's performance exactly (as seen in notebook 04). Thus the pipeline reproduces the original model behaviour with no data leakage or feature mismatch.

**APPLYING TUNED THRESHOLD :**

In [7]:
#apply F2 optimized threshold instead of the default 0.5
BEST_THRESHOLD = 0.341

def predict_with_threshold(pipeline, X, threshold=BEST_THRESHOLD):
    proba = pipeline.predict_proba(X)[:, 1]
    pred = (proba >= threshold).astype(int)
    return pred, proba

y_pred_thresh, y_proba_thresh = predict_with_threshold(pipeline, X_test_raw)
print(classification_report(y_test, y_pred_thresh))

              precision    recall  f1-score   support

           0       0.94      0.78      0.85      1593
           1       0.48      0.80      0.60       407

    accuracy                           0.78      2000
   macro avg       0.71      0.79      0.73      2000
weighted avg       0.84      0.78      0.80      2000



The optimized F2 threshold (chosen in notebook 05) is directly applied to predict_proba's output, instead of the default 0.5.

The model's precision drops to 48% while recall rises to 80% for the churn class, consistent with the performance in notebook 05.


**SAVING FINAL PIPELINE :**

The fitted pipeline is saved as a single artifact so it can be loaded and used on new customer data without rerunning the code.


In [8]:
joblib.dump({'pipeline': pipeline, 'threshold': BEST_THRESHOLD}, 'churn_pipeline_final.pkl')

['churn_pipeline_final.pkl']

 **BUILDING SHAP EXPLAINER :**

The SHAP explainer is built on the pipeline's engineering and scaling steps applied to a background sample of raw training data, then saved separately for per-prediction explanations.

In [9]:
raw_names = preprocessor.get_feature_names_out()
clean_names = [n.split('__')[-1] for n in raw_names]

X_train_transformed = pipeline.named_steps['scale'].transform(pipeline.named_steps['engineer'].transform(X_train_raw))

X_train_transformed = pd.DataFrame(X_train_transformed, columns=clean_names)

background = X_train_transformed.sample(n=100, random_state=42)

explainer = shap.TreeExplainer(pipeline.named_steps['classifier'], background)
joblib.dump(explainer, 'shap_explainer.pkl')

['shap_explainer.pkl']

Verify predictions:

In [10]:
churner_idx = y_test[y_test == 1].index[0]
non_churner_idx = y_test[y_test == 0].index[0]

for label, idx in [('Churner', churner_idx), ('Non-churner', non_churner_idx)]:
    row = X_test_raw.loc[[idx]]
    pred, proba = predict_with_threshold(pipeline, row)
    print(f"{label} (actual={y_test.loc[idx]}): predicted={pred[0]}, probability={proba[0]:.3f}")

Churner (actual=1): predicted=1, probability=0.839
Non-churner (actual=0): predicted=0, probability=0.040


The pipeline correctly predicts a high churn probability (0.839) for a confirmed churner and a low probability (0.040) for a confirmed non-churner. Thus it behaves consistently from end-to-end.
